# Verify Data

Layer 1 verification: confirm that OHLCV bars in the ParquetDataCatalog
match the source exchange and contain no corruption, gaps, or duplicates.

Run this once per instrument × timeframe after ingesting data, and again
after any changes to the data pipeline.

See `VERIFICATION.md` for the full verification plan.

In [ ]:
# ── Cell 1: Imports + Config ──────────────────────────────────────
#
# ★ Change these two for each instrument / timeframe you verify. ★

import httpx
import pandas as pd
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.core import BINANCE_FUTURES_API_URL, INTERVAL_TO_BAR_SPEC, bar_type_str

INSTRUMENT_ID = "BTCUSDT-PERP.BINANCE"
BAR_INTERVAL  = "4h"

# Standard
BAR_TYPE_STR     = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
EXCHANGE_SOURCE  = INSTRUMENT_ID.split(".")[-1].lower()       # "binance"
BINANCE_SYMBOL   = INSTRUMENT_ID.split("-")[0]                 # "BTCUSDT"
BINANCE_API_URL  = BINANCE_FUTURES_API_URL                     # futures, not spot

_UNIT_TO_HOURS = {"MINUTE": 1 / 60, "HOUR": 1, "DAY": 24}
_step, _agg = INTERVAL_TO_BAR_SPEC[BAR_INTERVAL]
BAR_INTERVAL_HOURS = _step * _UNIT_TO_HOURS[_agg]

# Files
CATALOG_PATH = "../data/catalog"

print(f"Verifying: {INSTRUMENT_ID}  {BAR_INTERVAL}")

In [ ]:
# ── Cell 2: Load bars ──────────────────────────────────────────────

catalog = ParquetDataCatalog(CATALOG_PATH)
bars = catalog.bars(bar_types=[BAR_TYPE_STR])

first_ts = pd.Timestamp(bars[0].ts_event, unit="ns", tz="UTC")
last_ts = pd.Timestamp(bars[-1].ts_event, unit="ns", tz="UTC")

print(f"Bars loaded: {len(bars):,}")
print(f"Range:       {first_ts:%Y-%m-%d %H:%M} → {last_ts:%Y-%m-%d %H:%M}")

In [ ]:
# ── Cell 3: Lookahead bias check ──────────────────────────────────
#
# ts_event = bar open time.  ts_init = when NT "sees" the bar.
# For external bars (exchange-provided), ts_init must be at or after
# bar close time.  If ts_init == ts_event, NT sees the bar at open
# time and your strategy trades on a bar that hasn't closed yet.

expected_delta_ns = BAR_INTERVAL_HOURS * 3_600_000_000_000

first = bars[0]
delta_ns = first.ts_init - first.ts_event

print(f"ts_event:       {first.ts_event}")
print(f"ts_init:        {first.ts_init}")
print(f"Delta:          {delta_ns / 1e9:.0f}s  (expected {expected_delta_ns / 1e9:.0f}s)")

if delta_ns == expected_delta_ns:
    print("✅ ts_init_delta correct — no lookahead bias")
elif delta_ns == 0:
    print("❌ ts_init == ts_event — strategy sees bars before they close!")
else:
    print(f"⚠️  Unexpected delta: {delta_ns}ns — investigate")

In [ ]:
# ── Cell 4: Bar count sanity ──────────────────────────────────────
#
# Crypto trades 24/7, so the expected bar count is deterministic.
# A significant shortfall means bars were dropped during ingestion.
# A surplus means duplicates slipped through.

hours_spanned = (last_ts - first_ts).total_seconds() / 3600
expected_bars = int(hours_spanned / BAR_INTERVAL_HOURS) + 1

actual = len(bars)
diff = actual - expected_bars
pct_diff = abs(diff) / expected_bars * 100

print(f"Expected bars:  {expected_bars:,}")
print(f"Actual bars:    {actual:,}")
print(f"Difference:     {diff:+d} ({pct_diff:.2f}%)")

if pct_diff < 0.5:
    print("✅ Bar count within tolerance")
else:
    print(f"⚠️  Bar count off by {pct_diff:.1f}% — investigate")

In [ ]:
# ── Cell 5: Duplicate and gap detection ───────────────────────────

timestamps = pd.Series([b.ts_event for b in bars])

# ── Duplicates ──
dupes = timestamps.duplicated().sum()
print(f"Duplicate timestamps: {dupes}")
if dupes > 0:
    dupe_ts = timestamps[timestamps.duplicated(keep=False)]
    print("❌ Duplicate bars found:")
    for ts in dupe_ts.unique()[:5]:
        print(f"  {pd.Timestamp(ts, unit='ns', tz='UTC')}")
else:
    print("✅ No duplicates")

# ── Gaps ──
diffs_ns = timestamps.diff().dropna()
expected_ns = BAR_INTERVAL_HOURS * 3_600_000_000_000

# Allow 50% jitter to catch real gaps without flagging rounding
gaps = diffs_ns[diffs_ns > expected_ns * 1.5]

print(f"\nExpected interval: {expected_ns / 1e9:.0f}s")
print(f"Gaps found:        {len(gaps)}")

if len(gaps) > 0:
    print("\nLargest gaps:")
    for idx in gaps.nlargest(10).index:
        ts_before = pd.Timestamp(bars[idx - 1].ts_event, unit="ns", tz="UTC")
        ts_after = pd.Timestamp(bars[idx].ts_event, unit="ns", tz="UTC")
        gap_hours = float(gaps[idx]) / 3_600_000_000_000
        print(f"  {ts_before} → {ts_after}  ({gap_hours:.1f}h gap)")
else:
    print("✅ No gaps detected")

In [ ]:
# ── Cell 6: OHLCV invariant checks ───────────────────────────────
#
# Every bar must satisfy: L <= O <= H, L <= C <= H, all > 0, V >= 0.
# Violations mean corrupt data passed ingestion.

issues = []
for i, bar in enumerate(bars):
    o, h, l, c = float(bar.open), float(bar.high), float(bar.low), float(bar.close)
    v = float(bar.volume)

    if h < l:
        issues.append((i, "high < low", f"H={h} L={l}"))
    if o > h or o < l:
        issues.append((i, "open outside H/L", f"O={o} H={h} L={l}"))
    if c > h or c < l:
        issues.append((i, "close outside H/L", f"C={c} H={h} L={l}"))
    if o <= 0 or h <= 0 or l <= 0 or c <= 0:
        issues.append((i, "zero/negative price", f"O={o} H={h} L={l} C={c}"))
    if v < 0:
        issues.append((i, "negative volume", f"V={v}"))

print(f"Checked {len(bars):,} bars for OHLCV invariants")
if issues:
    print(f"❌ {len(issues)} issues found:")
    for idx, issue, detail in issues[:10]:
        ts = pd.Timestamp(bars[idx].ts_event, unit="ns", tz="UTC")
        print(f"  [{idx}] {ts} — {issue}: {detail}")
    if len(issues) > 10:
        print(f"  ... and {len(issues) - 10} more")
else:
    print("✅ All bars pass OHLCV sanity checks")

In [ ]:
# ── Cell 7: Spot-check against exchange API ──────────────────────
#
# Pull candles directly from the exchange and compare OHLC values
# to what's in the catalog.  This is the definitive Layer 1 check:
# does stored data match the source of truth?
#
# Volume comparison is informational only — exchanges report volume
# in different denominations (base vs quote vs contracts).
#
# If you can't reach the exchange API from this environment, do the
# comparison manually against TradingView and note results in a
# markdown cell below.


def fetch_binance_kline(symbol: str, interval: str, open_time_ms: int) -> dict:
    """Fetch a single kline from Binance Futures API."""
    resp = httpx.get(
        f"{BINANCE_API_URL}/fapi/v1/klines",
        params={
            "symbol": symbol,
            "interval": interval,
            "startTime": open_time_ms,
            "limit": 1,
        },
    )
    resp.raise_for_status()
    k = resp.json()[0]
    return {
        "open": float(k[1]),
        "high": float(k[2]),
        "low": float(k[3]),
        "close": float(k[4]),
        "volume": float(k[5]),
    }


# Pick bars at different points in the dataset
check_indices = [
    100,                    # early
    len(bars) // 4,         # 25%
    len(bars) // 2,         # midpoint
    len(bars) * 3 // 4,     # 75%
    len(bars) - 100,        # late
]

print(f"Spot-checking {len(check_indices)} bars against {EXCHANGE_SOURCE} API\n")
mismatches = 0
checked = 0

for idx in check_indices:
    bar = bars[idx]
    bar_ts = pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")
    open_time_ms = int(bar_ts.timestamp() * 1000)

    catalog_ohlcv = {
        "open": float(bar.open),
        "high": float(bar.high),
        "low": float(bar.low),
        "close": float(bar.close),
        "volume": float(bar.volume),
    }

    try:
        if EXCHANGE_SOURCE == "binance":
            exchange_ohlcv = fetch_binance_kline(
                BINANCE_SYMBOL, BAR_INTERVAL, open_time_ms
            )
        else:
            print(f"  ⚠️  [{idx}] {bar_ts} — {EXCHANGE_SOURCE} API not implemented")
            print(f"       Compare manually on TradingView:")
            print(f"       O={catalog_ohlcv['open']}  H={catalog_ohlcv['high']}  "
                  f"L={catalog_ohlcv['low']}  C={catalog_ohlcv['close']}")
            continue
    except Exception as e:
        print(f"  ⚠️  [{idx}] {bar_ts} — API error: {e}")
        continue

    # Compare OHLC (skip volume — denomination varies by source)
    match = all(
        abs(catalog_ohlcv[k] - exchange_ohlcv[k]) < 0.01
        for k in ["open", "high", "low", "close"]
    )

    checked += 1
    if not match:
        mismatches += 1

    status = "✅" if match else "❌"
    print(f"  {status} [{idx}] {bar_ts}")
    if not match:
        for k in ["open", "high", "low", "close", "volume"]:
            c = catalog_ohlcv[k]
            e = exchange_ohlcv[k]
            flag = " ← MISMATCH" if k != "volume" and abs(c - e) >= 0.01 else ""
            print(f"       {k:>6}: catalog={c:<14} exchange={e:<14}{flag}")

if checked == 0:
    print(f"\n⚠️  No bars checked — API unreachable. Compare manually on TradingView.")
elif mismatches == 0:
    print(f"\n✅ All {checked} checked bars match exchange")
else:
    print(f"\n❌ {mismatches}/{checked} mismatches found")

### Manual TradingView spot-check (if API unavailable)

If the API cell above can't reach the exchange, manually compare these bars
on TradingView and record results here:

| Bar index | Timestamp (UTC) | Catalog OHLC | TradingView OHLC | Match? |
|-----------|----------------|--------------|------------------|--------|
| | | | | |
| | | | | |
| | | | | |

In [ ]:
# ── Cell 8: Summary ───────────────────────────────────────────────

print("=" * 60)
print(f"  DATA VERIFICATION SUMMARY")
print(f"  {INSTRUMENT_ID}  {BAR_INTERVAL}")
print(f"  {first_ts:%Y-%m-%d} → {last_ts:%Y-%m-%d}  ({len(bars):,} bars)")
print("=" * 60)

checks = []

# Lookahead
if delta_ns == expected_delta_ns:
    checks.append(("✅", "Lookahead bias", "ts_init_delta correct"))
else:
    checks.append(("❌", "Lookahead bias", f"ts_init_delta = {delta_ns}ns"))

# Bar count
if pct_diff < 0.5:
    checks.append(("✅", "Bar count", f"{actual:,} bars ({pct_diff:.2f}% off expected)"))
else:
    checks.append(("⚠️", "Bar count", f"{actual:,} bars ({pct_diff:.1f}% off expected)"))

# Duplicates
if dupes == 0:
    checks.append(("✅", "Duplicates", "None"))
else:
    checks.append(("❌", "Duplicates", f"{dupes} found"))

# Gaps
if len(gaps) == 0:
    checks.append(("✅", "Gaps", "None"))
else:
    checks.append(("⚠️", "Gaps", f"{len(gaps)} gaps found"))

# OHLCV invariants
if len(issues) == 0:
    checks.append(("✅", "OHLCV invariants", "All bars valid"))
else:
    checks.append(("❌", "OHLCV invariants", f"{len(issues)} violations"))

# Exchange spot-check
if checked == 0:
    checks.append(("⚠️", "Exchange spot-check", "API unreachable — not verified"))
elif mismatches == 0:
    checks.append(("✅", "Exchange spot-check", f"All {checked} sampled bars match"))
else:
    checks.append(("❌", "Exchange spot-check", f"{mismatches}/{checked} mismatches"))

for icon, label, detail in checks:
    print(f"  {icon} {label:<22} {detail}")

all_pass = all(icon == "✅" for icon, _, _ in checks)
print()
if all_pass:
    print("  VERDICT: ✅ Data verified — proceed to indicator checks.")
else:
    print("  VERDICT: ⚠️  Issues found — investigate before trusting backtests.")
print("=" * 60)